# RAE JAX CelebA-HQ Kaggle TPU v5e-8 Pipeline (SiTDH-B)

Notebook này là bản sao `SiTDH-B` đã chỉnh riêng cho Kaggle `TPU v5e-8` với host `96 vCPU`.

Flow chính:

- clone repo rồi checkout branch `jax-sit-dh-celebahq256`
- đồng bộ dependency từ `pyproject.toml` vào `/tmp/.venv` bằng `uv sync`
- tự xóa cờ executable-stack mà Kaggle có thể chặn trên `jaxlib`, rồi mới import JAX qua `uv run`
- tải `eurecom-ds/celeba-hq-256` từ Hugging Face rồi export thành `ImageFolder` `256x256`
- build `stage_1` latent stat trên TPU
- reconstruct Stage 1 trên TPU
- build backend-native FID reference stats bằng đúng detector JAX của backend
- ghi config Stage 2 JAX `SiTDH-B` cho TPU, với validation loss và online FID đều đi qua backend JAX

Lưu ý:

- notebook dùng Hugging Face dataset `eurecom-ds/celeba-hq-256`, nên không còn phụ thuộc `manual_dir` hay bộ tar thủ công của TFDS
- cell lấy secret Kaggle và cell display vẫn dùng kernel Kaggle gốc
- các bước phụ thuộc package như check JAX, export dữ liệu, build stat, reconstruction, FID, và train đều đi qua môi trường `uv` tại `/tmp/.venv`
- notebook sẽ gọi `scripts/clear_elf_execstack.py --package jaxlib` để xử lý lỗi `cannot enable executable stack` nếu Kaggle chặn `jaxlib`
- notebook này ghim `transformers==4.57.1`; repo sẽ tự vá cache `diffuse_nnx` để import Dinov2 từ subpackage tương thích của `transformers` và lazy-load `google.cloud.storage` chỉ khi backend thực sự cần tải asset từ GCS
- nếu vừa cài hoặc reinstall `jax` / `jaxlib` trong notebook, không nên import trực tiếp trong kernel đang chạy khi chưa restart

Giới hạn hiện tại:

- branch `jax-sit-dh-celebahq256` vẫn giữ pipeline train JAX hiện tại nhận `ImageFolder`; notebook chỉ đổi nguồn download sang Hugging Face rồi materialize dữ liệu đầu vào tại chỗ
- notebook này tối ưu cho `TPU v5e-8`; nếu bạn chuyển sang GPU thì nên dùng notebook gốc


In [ ]:
%cd /kaggle/working
!rm -rf RAE
!git clone https://github.com/sontungkieu/RAE
%cd /kaggle/working/RAE
!git checkout jax-sit-dh-celebahq256
!curl -LsSf https://astral.sh/uv/install.sh | sh
!ln -sf /root/.local/bin/uv /usr/local/bin/uv


In [ ]:
import os

os.environ["UV_PROJECT_ENVIRONMENT"] = "/tmp/.venv"
os.environ["UV_CACHE_DIR"] = "/tmp/uv-cache"

!uv sync -q
!uv run python scripts/clear_elf_execstack.py --package jaxlib
print("Synced the repo dependencies into /tmp/.venv. The package-backed steps below all go through uv run.")


In [ ]:
import os
import pathlib

os.environ["JAX_PLATFORMS"] = "tpu,cpu"
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"
os.environ["PYOPENGL_PLATFORM"] = "egl"

try:
    from kaggle_secrets import UserSecretsClient

    secrets = UserSecretsClient()
    wandb_token = secrets.get_secret("WANDB2")
    hf_token = secrets.get_secret("HF_TOK_WRITE_KAGGLE")

    os.environ["WANDB_API_KEY"] = wandb_token
    os.environ["WANDB_KEY"] = wandb_token
    os.environ["HF_TOKEN"] = hf_token

    netrc = pathlib.Path.home() / ".netrc"
    netrc.write_text(f"machine api.wandb.ai login user password {wandb_token}\n")
    os.chmod(netrc, 0o600)
    print("Loaded Kaggle secrets for wandb and Hugging Face.")
except Exception as exc:
    print(f"Skipping Kaggle secret bootstrap: {exc}")


In [ ]:
%%bash
set -euo pipefail

cd /kaggle/working/RAE

uv run python scripts/clear_elf_execstack.py --package jaxlib --quiet-unchanged

export JAX_PLATFORMS="tpu,cpu"
export XLA_PYTHON_CLIENT_PREALLOCATE="false"

uv run python - <<'PY'
import os
import sys

import jax
import jaxlib

print("python executable:", sys.executable)
print("jax version:", jax.__version__)
print("jaxlib version:", jaxlib.__version__)
print("host cpu cores:", os.cpu_count())
print("default backend:", jax.default_backend())
print("local device count:", jax.local_device_count())
print("devices:", jax.devices())
PY


In [ ]:
%%bash
set -euo pipefail

cd /kaggle/working/RAE

uv run hf download nyu-visionx/RAE-collections \
  decoders/dinov2/wReg_base/ViTXL_n08/model.pt \
  --local-dir models


In [ ]:
from pathlib import Path

repo_root = Path("/kaggle/working/RAE")
hf_cache_dir = Path("/kaggle/working/hf_datasets_cache")
celebahq_root = Path("/kaggle/working/celebahq256_imgfolder")
stage1_cfg_path = repo_root / "configs" / "stage1" / "pretrained" / "CelebAHQ256_DINOv2-B_jax_tpuv5e8.yaml"
stage2_cfg_path = repo_root / "configs" / "stage2" / "training" / "CelebAHQ256_SiTDH-B_DINOv2-B_jax_tpuv5e8.yaml"
bootstrap_stats_path = Path("/kaggle/working/bootstrap_identity_stat.pt")
latent_stats_path = Path("/kaggle/working/celebahq256_stage1_latent_stat_tpu.pt")
fid_stats_path = Path("/kaggle/working/celebahq256_val_fid_stats_cpu.pkl")
stage1_single_recon_path = Path("/kaggle/working/celebahq256_stage1_single_recon_tpu.png")
stage1_recon_dir = Path("/kaggle/working/celebahq256_stage1_recon_val_tpu")
stage2_results_dir = Path("/kaggle/working/results_jax_tpu")

stats_batch_size = 64
stats_num_workers = 16
recon_batch_size = 16
recon_num_workers = 16
fid_num_workers = 32
recon_limit = 2048  # bỏ limit nếu muốn reconstruct toàn bộ val split

print("repo_root:", repo_root)
print("hf_cache_dir:", hf_cache_dir)
print("celebahq_root:", celebahq_root)
print("stage1_cfg_path:", stage1_cfg_path)
print("stage2_cfg_path:", stage2_cfg_path)


In [ ]:
%%bash
set -euo pipefail

cd /kaggle/working/RAE

uv run python src_jax/export_celebahq_hf.py \
  --dataset eurecom-ds/celeba-hq-256 \
  --cache-dir /kaggle/working/hf_datasets_cache \
  --output /kaggle/working/celebahq256_imgfolder

uv run python - <<'PYSUM'
import json
from pathlib import Path

summary_path = Path("/kaggle/working/celebahq256_imgfolder/hf_export_summary.json")
summary = json.loads(summary_path.read_text())
print(json.dumps(summary, indent=2))
PYSUM


In [ ]:
%%bash
set -euo pipefail

cd /kaggle/working/RAE

uv run python - <<'PY'
from pathlib import Path
repo_root = Path("/kaggle/working/RAE")
celebahq_root = Path("/kaggle/working/celebahq256_imgfolder")
stage1_cfg_path = repo_root / "configs" / "stage1" / "pretrained" / "CelebAHQ256_DINOv2-B_jax_tpuv5e8.yaml"
stage2_cfg_path = repo_root / "configs" / "stage2" / "training" / "CelebAHQ256_SiTDH-B_DINOv2-B_jax_tpuv5e8.yaml"
bootstrap_stats_path = Path("/kaggle/working/bootstrap_identity_stat.pt")
latent_stats_path = Path("/kaggle/working/celebahq256_stage1_latent_stat_tpu.pt")
fid_stats_path = Path("/kaggle/working/celebahq256_val_fid_stats_cpu.pkl")

import textwrap

import torch

bootstrap_stats_path.parent.mkdir(parents=True, exist_ok=True)
torch.save(
    {
        "mean": torch.zeros((1, 1, 1), dtype=torch.float32),
        "var": torch.ones((1, 1, 1), dtype=torch.float32),
        "count": 0,
    },
    bootstrap_stats_path,
)

stage1_cfg_text = textwrap.dedent(
    f"""
    stage_1:
      target: stage1.RAE
      params:
        encoder_cls: 'Dinov2withNorm'
        encoder_config_path: 'facebook/dinov2-with-registers-base'
        encoder_input_size: 224
        encoder_params:
          dinov2_path: 'facebook/dinov2-with-registers-base'
          normalize: true
        decoder_config_path: 'configs/decoder/ViTXL'
        pretrained_decoder_path: 'models/decoders/dinov2/wReg_base/ViTXL_n08/model.pt'
        noise_tau: 0.0
        reshape_to_2d: true
        normalization_stat_path: '{latent_stats_path.as_posix()}'
    """
).strip() + "\n"

stage2_cfg_text = textwrap.dedent(
    f"""
    stage_1:
      target: stage1.RAE
      ckpt: null
      params:
        encoder_cls: 'Dinov2withNorm'
        encoder_config_path: 'facebook/dinov2-with-registers-base'
        encoder_input_size: 224
        encoder_params:
          dinov2_path: 'facebook/dinov2-with-registers-base'
          normalize: true
        decoder_config_path: 'configs/decoder/ViTXL'
        pretrained_decoder_path: 'models/decoders/dinov2/wReg_base/ViTXL_n08/model.pt'
        noise_tau: 0.0
        reshape_to_2d: true
        normalization_stat_path: '{latent_stats_path.as_posix()}'

    stage_2:
      target: stage2.models.SiT.SiTDH
      ckpt: null
      params:
        input_size: 16
        patch_size: 1
        in_channels: 768
        hidden_size: [768, 2048]
        depth: [12, 2]
        num_heads: [12, 16]
        mlp_ratio: 4.0
        class_dropout_prob: 0.0
        num_classes: 1
        use_qknorm: false
        use_swiglu: true
        use_rope: true
        use_rmsnorm: true
        use_pos_embed: true
        wo_shift: false

    transport:
      params:
        path_type: 'Linear'
        prediction: 'velocity'
        loss_weight: null
        time_dist_type: 'uniform'

    sampler:
      mode: ODE
      params:
        sampling_method: 'euler'
        num_steps: 50
        atol: 1.0e-6
        rtol: 1.0e-3
        reverse: false

    guidance:
      method: 'cfg'
      scale: 1.0
      t_min: 0.0
      t_max: 1.0

    misc:
      latent_size: [768, 16, 16]
      num_classes: 1
      time_dist_shift_dim: 196608
      time_dist_shift_base: 4096

    eval:
      data_path: '{(celebahq_root / "val").as_posix()}'
      eval_every: 5000
      batch_size: 4
      num_workers: 8
      max_batches: 32
      eval_model: false
      fid_ref: '{fid_stats_path.as_posix()}'
      fid_every: 5000
      fid_num_samples: 4096
      fid_per_proc_batch_size: 4
      fid_batch_size: 128

    training:
      global_seed: 0
      epochs: 200
      global_batch_size: 128
      grad_accum_steps: 1
      ema_decay: 0.9995
      num_workers: 8
      random_flip: true
      log_every: 10
      ckpt_every: 210000
      sample_every: 5000
      base_lr: 0.0001
      final_lr: 0.00001
      beta: [0.9, 0.95]
      wd: 0.0
      schedule_type: 'linear'
      decay_start_epoch: 150
      decay_end_epoch: 200
      clip_grad: 1.0
    """
).strip() + "\n"

stage1_cfg_path.parent.mkdir(parents=True, exist_ok=True)
stage2_cfg_path.parent.mkdir(parents=True, exist_ok=True)
stage1_cfg_path.write_text(stage1_cfg_text, encoding="utf-8")
stage2_cfg_path.write_text(stage2_cfg_text, encoding="utf-8")

print(f"Wrote {stage1_cfg_path}")
print(f"Wrote {stage2_cfg_path}")
print(f"Bootstrap stats path: {bootstrap_stats_path}")
PY


In [ ]:
%%bash
set -euo pipefail

cd /kaggle/working/RAE

uv run python scripts/clear_elf_execstack.py --package jaxlib --quiet-unchanged

uv run python src_jax/build_stage1_stats.py \
  --config configs/stage1/pretrained/CelebAHQ256_DINOv2-B_jax_tpuv5e8.yaml \
  --input /kaggle/working/celebahq256_imgfolder/train \
  --output /kaggle/working/celebahq256_stage1_latent_stat_tpu.pt \
  --batch-size 64 \
  --num-workers 16 \
  --set stage_1.params.normalization_stat_path=/kaggle/working/bootstrap_identity_stat.pt


In [ ]:
%%bash
set -euo pipefail

cd /kaggle/working/RAE

uv run python - <<'PY'
from pathlib import Path
latent_stats_path = Path("/kaggle/working/celebahq256_stage1_latent_stat_tpu.pt")
celebahq_root = Path("/kaggle/working/celebahq256_imgfolder")

import torch

stats = torch.load(latent_stats_path, map_location="cpu")
print("latent mean shape:", tuple(stats["mean"].shape))
print("latent var shape:", tuple(stats["var"].shape))
print("num samples:", stats["count"])

sample_image_path = next(path for path in (celebahq_root / "val" / "face").iterdir() if path.is_file())
print("sample image:", sample_image_path)
PY


In [ ]:
%%bash
set -euo pipefail

cd /kaggle/working/RAE

uv run python scripts/clear_elf_execstack.py --package jaxlib --quiet-unchanged

sample_image=$(find /kaggle/working/celebahq256_imgfolder/val/face -type f -print -quit)
[ -n "${sample_image}" ]

uv run python src_jax/stage1_sample.py \
  --config configs/stage1/pretrained/CelebAHQ256_DINOv2-B_jax_tpuv5e8.yaml \
  --image "${sample_image}" \
  --output /kaggle/working/celebahq256_stage1_single_recon_tpu.png


In [ ]:
from IPython.display import Image, display
display(Image(filename=stage1_single_recon_path))

In [ ]:
%%bash
set -euo pipefail

cd /kaggle/working/RAE

uv run python scripts/clear_elf_execstack.py --package jaxlib --quiet-unchanged

uv run python src_jax/reconstruct_folder.py \
  --config configs/stage1/pretrained/CelebAHQ256_DINOv2-B_jax_tpuv5e8.yaml \
  --input /kaggle/working/celebahq256_imgfolder/val \
  --output-dir /kaggle/working/celebahq256_stage1_recon_val_tpu \
  --batch-size 16 \
  --num-workers 16 \
  --limit 2048


In [ ]:
%%bash
set -euo pipefail

cd /kaggle/working/RAE

uv run python src_jax/build_fid_stats.py \
  --input /kaggle/working/celebahq256_imgfolder/val \
  --output /kaggle/working/celebahq256_val_fid_stats_cpu.pkl \
  --image-size 256 \
  --batch-size 64 \
  --num-workers 32


## Optional: launch Stage 2 JAX SiTDH-B training on Kaggle TPU v5e-8

Cell dưới đây giữ train, validation loss, và online FID trong cùng backend JAX trên TPU với config CelebA-HQ vừa ghi ở trên.


In [ ]:
%%bash
set -euo pipefail

cd /kaggle/working/RAE

uv run python scripts/clear_elf_execstack.py --package jaxlib --quiet-unchanged

timestamp="$(TZ=Asia/Bangkok date +%Y%m%d-%H%M%S)"
run_name="CelebAHQ256_SiTDH-B_DINOv2-B_jax_tpuv5e8-${timestamp}"

export ENTITY="TungBangDSLab"
export PROJECT="rae-jax-celebahq256-tpuv5e8-sitdh-b"

uv run python src_jax/train.py \
  --config configs/stage2/training/CelebAHQ256_SiTDH-B_DINOv2-B_jax_tpuv5e8.yaml \
  --data-path /kaggle/working/celebahq256_imgfolder \
  --results-dir /kaggle/working/results_jax_tpu \
  --precision bf16 \
  --exp-name "${run_name}" \
  --wandb \
  --wandb-entity TungBangDSLab \
  --wandb-project "${PROJECT}" \
  --set training.global_batch_size=64 \
  --set training.num_workers=16 \
  --set training.prefetch_factor=4 \
  --set training.ckpt_every=210000 \
  --set training.log_rae_latent_stats=true \
  --set training.log_activation_stats=true \
  --set eval.prefetch_factor=4 \
  --set eval.fid_ref=/kaggle/working/celebahq256_val_fid_stats_cpu.pkl \
  --set eval.fid_every=10000 \
  --set eval.fid_num_samples=4096
